# AtlasRank M4 full-catalog Colab execution — UNAPPROVED

Preparation only. Do not connect a GPU or run this notebook until the lead approves the exact source/export, full manifest, private Drive destination, exclusive ownership and cumulative prior allocation charge. Separate Mac import/evaluation approval is required. No judgments, training, M5, Git remote or full-catalog CPU fallback.

Follow `docs/M4_FULL_RUN.md`. All ML, validation and checkpoint logic lives in the package. Preserve every earlier pilot/runtime artifact. Record the runtime connection UTC epoch **before uploads/setup**, including idle time; never reset it on a same-runtime retry.


In [ ]:
# Set only AFTER explicit full-run authorization. Missing values intentionally stop execution.
from pathlib import Path

ALLOCATION_STARTED_UTC = None  # Actual UTC epoch when this GPU runtime was allocated.
APPROVED_MANIFEST = Path("/content/m4-colab-full-run.approved.json")
ARCHIVE = Path("/content/atlasrank-full-input-v4.zip")
INPUTS = Path("/content/atlasrank-full-input-v4")
WORK = Path("/content/atlasrank-full-work")
if ALLOCATION_STARTED_UTC is None:
    raise PermissionError("Full authorization and actual allocation timestamp required")

In [ ]:
import sys

import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable; no fallback")
print("GPU:", torch.cuda.get_device_name(0))
print("Free/total VRAM:", torch.cuda.mem_get_info(0))
if (
    sys.version_info[:2] != (3, 13)
    or str(torch.__version__) != "2.11.0+cu130"
    or torch.version.cuda != "13.0"
    or torch.cuda.get_device_name(0) != "NVIDIA L4"
):
    raise RuntimeError("Runtime differs; stop for compatibility review")
# The package also checks exact accepted driver/CUDA libraries and encoder recipe.

Upload only the reviewed ZIP and the separately approved ignored JSON. The ZIP contains 1,215,854 label-free canonical identities/titles, required model files and source/provenance. It contains no query or relevance data. The bootstrap below executes only the hash-verified archive's standard-library transport module; it does not install or replace Torch.


In [ ]:
import hashlib
import json
import zipfile

EXPECTED_ZIP_SHA256 = "e8f23d1fa32df69f48617e3fd31e7c32767c48076336d55d7d6afe9c24d3e87e"
if ARCHIVE.stat().st_size > 512 * 1024**2:
    raise ValueError("Upload exceeds limit")
h = hashlib.sha256()
with ARCHIVE.open("rb") as f:
    for block in iter(lambda: f.read(1024**2), b""):
        h.update(block)
if h.hexdigest() != EXPECTED_ZIP_SHA256:
    raise ValueError("Reviewed ZIP mismatch")
approved = json.loads(APPROVED_MANIFEST.read_text())
if not approved.get("execution_approved") or not approved.get("runnable"):
    raise PermissionError("An unapproved proposal cannot run")
with zipfile.ZipFile(ARCHIVE) as z:
    bootstrap = z.read("source/src/atlasrank/retrieval/colab_transport.py")
namespace = {"__name__": "atlasrank_verified_bootstrap"}
exec(compile(bootstrap, "<hash-verified transport bootstrap>", "exec"), namespace)
# Fresh destination only: preserve earlier attempts; do not silently overwrite.
namespace["safe_extract"](ARCHIVE, INPUTS, 512 * 1024**2)
sys.path.insert(0, str(INPUTS / "source/src"))
from atlasrank.retrieval.colab_full_contract import approve_full  # noqa: E402

persistent = Path(approved["persistent_destination"])
approve_full(APPROVED_MANIFEST, INPUTS, persistent)
print("Reviewed export/source/configuration verified; labels absent")

In [ ]:
import os

from google.colab import drive

# Human confirms the approved private account in Google's mount dialog.
drive.mount("/content/drive")
if not persistent.parent.is_dir():
    raise RuntimeError("Approved Drive parent missing")
# Stop the previous runtime first. Never run two writers or steal ownership.
os.environ["ATLAS_COLAB_EXCLUSIVE_RUNTIME"] = approved["operator_owner_token"]
os.environ["PYTHONPATH"] = str(INPUTS / "source/src")

In [ ]:
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "atlasrank.retrieval.colab_full_runner",
        "--manifest",
        str(APPROVED_MANIFEST),
        "--inputs",
        str(INPUTS),
        "--persistent",
        str(persistent),
        "--work",
        str(WORK),
        "--allocation-started-utc",
        str(ALLOCATION_STARTED_UTC),
    ],
    check=True,
)
# Package supervisor handles pinned overlay/bootstrap, CUDA checks, encoding,
# immutable receipt publication, progress, cumulative charges and return packaging.

After success, download the return ZIP, matching transfer JSON and final session `.end.json`. The final supervisor end record is written after ZIP creation, so it is a separate required sidecar. Preserve all Drive contents. Check downloads on the Mac, record the actual disconnect timestamp/total allocation, then disconnect and delete this runtime. Managed supervision cannot stop billing after Python exits. Stop before the cumulative five-hour allocation limit even if downloads are incomplete; receipts remain in Drive.

On interruption: stop the old runtime, retain the identical approved manifest/destination/token and all receipts. Record each new runtime's connection timestamp. Re-upload/extract into a fresh VM and rerun under the remaining cumulative budget; validated receipt-backed chunks are restored and skipped. All gaps since the first full allocation, including disconnected time, are conservatively charged and may exhaust the budget. Do not change batch size, recipe, thresholds or limits to resume.


In [ ]:
from google.colab import files

returns = list(WORK.glob("*/full-return.zip"))
if len(returns) != 1:
    raise RuntimeError("Select the successful attempt explicitly; preserve all attempts")
result = returns[0]
sidecar = result.with_suffix(".transfer.json")
session_id = result.parent.name
end = persistent / (session_id + ".end.json")
if not sidecar.is_file() or not end.is_file():
    raise RuntimeError("Missing final transfer/session evidence")
for path in (result, sidecar, end):
    files.download(str(path))
print("Verify all three Mac downloads; record allocation/disconnect and release GPU promptly")